# Masking delle feature FC in funzione della lesione

>> **Prototipo** eseguito su 3 pazienti reali, prima che il metodo diventasse codice in `src/features/functional.py`. 

Teoria completa (cos'è una matrice FC, notazione `NodoA__NodoB`, letteratura di riferimento, decisioni di metodo, esempio numerico) in [`assets/knowledge/fc_lesion_masking.md`](../assets/knowledge/fc_lesion_masking.md).


**Cosa fa questo notebook, in breve**: esegue la procedura passo per passo descritta in [`assets/knowledge/fc_lesion_masking.md`](../assets/knowledge/fc_lesion_masking.md#parte-3--procedura-passo-per-passo) (Parte 3), organizzata in tre blocchi:

- **Setup** — carica l'atlante (una sola volta).
- **Step 1-8** — per ogni paziente, uno alla volta (ruolo di `mask_fc.py`): lesione → riallinea → coverage → nodi compromessi → carica FC → marca NaN → salva → vettorizza.
- **Step 9-12** — tutti i pazienti insieme (ruolo di `build_fc_matrix.py`): verifica etichette → impila → controllo qualità → salva.

Le celle qui sotto seguono questi blocchi. Per il *perché* di ogni scelta (NaN vs zero, due pipeline separate, ecc.) vedi la [Parte 2](../assets/knowledge/fc_lesion_masking.md#parte-2--decisioni-implementative-generali) del `.md`. Destinazioni su disco: `data/derived/features/masked_fc/` e `data/derived/features/fc_matrix/`.

In [ ]:
import warnings

# nilearn/nibabel emettono warning di deprecazione non rilevanti per questa analisi - silenziati
# solo per leggibilita' dell'output, non nascondono errori reali (quelli restano ValueError/AssertionError).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import glob  # per trovare il file CSV della matrice FC senza scrivere il nome esatto a mano
from pathlib import Path

import nibabel as nib  # legge/scrive immagini cerebrali in formato NIfTI (.nii.gz)
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img  # riallinea due immagini 3D sulla stessa griglia spaziale
from nilearn.maskers import NiftiLabelsMasker  # estrae valori per-zona da un'immagine, data una mappa di zone

# --- Parametri di questa run: piu' pazienti (dimostrativi), una versione della mappa cerebrale ---
SUBJECTS = ["sub-STUNIPD0002", "sub-STUNIPD0003", "sub-STUNIPD0006"]  # lesioni diverse, per mostrare pattern diversi di masking
ATLAS_COMBO = "atlas-Yan200TianS2Buckner7N"  # una delle 12 mappe cerebrali disponibili (200 zone corticali + subcortex + cervelletto)
MIN_COVERAGE = 0.5  # soglia: sotto il 50% di territorio sano, la zona e' "compromessa" (standard di campo, vedi sopra)
DEMO_OUTPUT_NAME = "demo"  # nome cartella di output - run dimostrativa su 3 pazienti, distinta dalla run reale su tutta la coorte

DATA_ROOT = "../data/clinical_connectome/derivatives/UNIPD/WashU"  # dati del paziente (lesione + connettivita')
ATLAS_ROOT = f"../assets/atlases/fmriprep/{ATLAS_COMBO}"  # mappa cerebrale di riferimento (copiata dal server)

# destinazioni finali concordate - create qui se non esistono ancora, mai assunte gia' presenti
MASKED_FC_ROOT = Path(f"../data/derived/features/masked_fc/{DEMO_OUTPUT_NAME}")
FC_MATRIX_ROOT = Path(f"../data/derived/features/fc_matrix/{DEMO_OUTPUT_NAME}")
MASKED_FC_ROOT.mkdir(parents=True, exist_ok=True)
FC_MATRIX_ROOT.mkdir(parents=True, exist_ok=True)

## Setup — Carica l'atlante

Una sola volta, non per paziente: assegna ogni voxel a uno dei ~239 nodi, serve per sapere quanto la lesione copre ciascuno. Da qui si ricava `node_names` (l'elenco ordinato dei nomi delle zone), riusato da tutti gli step successivi. Dettagli in [Parte 1, sezione 5](../assets/knowledge/fc_lesion_masking.md#5-latlante-mappa-cerebrale) del `.md`.

In [2]:
# squeeze_image: il file ha una dimensione extra inutile (4 dimensioni invece di 3), la togliamo
atlas_img = nib.squeeze_image(
    nib.load(f"{ATLAS_ROOT}/{ATLAS_COMBO}_space-MNI152NLin6Asym_res-2_dseg.nii.gz")
)

# la tabella che traduce ogni numero-zona nel suo nome (es. 1 -> "7Networks_LH_Default_IPL_1")
label_table = pd.read_csv(f"{ATLAS_ROOT}/{ATLAS_COMBO}_dseg.tsv", sep="\t")
label_ids = label_table["index"].tolist()  # tutti i numeri-zona attesi, nell'ordine ufficiale
id_to_name = dict(zip(label_table["index"], label_table["label"]))  # dizionario numero -> nome
node_names = np.array([id_to_name[i] for i in label_ids])  # nomi leggibili delle zone, nell'ordine ufficiale

print(f"Atlante caricato: {len(label_ids)} nodi")
label_table.head(3)  # anteprima: come si presenta la tabella

Atlante caricato: 239 nodi


,index,label
0,1,7Networks_LH_Default_IPL_1
1,2,7Networks_LH_Default_IPL_2
2,3,7Networks_LH_Default_IPL_3


## Step 1-8 — per ogni paziente (`mask_fc.py`)

Questi 8 step si ripetono identici per ogni paziente, uno alla volta. La cella sotto (`process_subject`) li esegue tutti in sequenza per ciascuno dei pazienti demo.

1. Carica la lesione.
2. Riallinea la lesione alla griglia dell'atlante (nearest neighbor).
3. Calcola la coverage (% di zona sana) per ogni nodo.
4. Decide quali nodi sono compromessi (coverage < soglia 50%).
5. Carica la matrice di connettività e verifica che i nomi combacino con l'atlante.
6. Marca con NaN righe/colonne dei nodi compromessi (NaN, non zero — il valore concreto si decide solo a valle).
7. Salva su disco la matrice mascherata, così com'è (unico output su disco di questo blocco).
8. Vettorizza (triangolo superiore) → un vettore *in memoria* per paziente, non salvato da solo: attende l'impilamento (step 10).

Esempio numerico passo-passo su atlante giocattolo in [Parte 3](../assets/knowledge/fc_lesion_masking.md#step-1-8--per-ogni-paziente-mask_fcpy) del `.md`.

In [3]:
def compute_parcel_coverage(atlas_img, label_ids, healthy_img):
    """Frazione di voxel sani per ciascuna zona, nello stesso ordine di label_ids.

    Una zona con zero voxel sani rimasti viene rimossa da NiftiLabelsMasker quando gli si
    passa mask_img (non restituita come 0) - gestito qui esplicitamente come coverage
    0.0, mai lasciato disallineare silenziosamente il conteggio mascherato da quello totale.
    """
    # Immagine "contatore": vale 1 in ogni punto del cervello. int32, non uint8: con una
    # casella troppo stretta il conteggio va in overflow silenzioso (bug verificato su dati reali,
    # vedi docs/debugging/debug_23_07_26.md).
    ones_img = nib.Nifti1Image(np.ones(atlas_img.shape, dtype=np.int32), atlas_img.affine, atlas_img.header)

    masker_total = NiftiLabelsMasker(labels_img=atlas_img, background_label=0, strategy="sum", standardize=False)
    n_total = np.squeeze(masker_total.fit_transform(ones_img))
    total_by_label = dict(zip(masker_total.labels_[1:], n_total))  # [1:]: labels_[0] e' il placeholder "Background"

    masker_healthy = NiftiLabelsMasker(
        labels_img=atlas_img, mask_img=healthy_img, background_label=0, strategy="sum", standardize=False
    )
    n_healthy = np.squeeze(masker_healthy.fit_transform(ones_img))
    healthy_by_label = dict(zip(masker_healthy.labels_[1:], np.atleast_1d(n_healthy)))

    # .get(lbl, 0.0): se una zona e' sparita dal secondo conteggio (100% lesionata), il suo valore e' 0.0 - mai un errore.
    return np.array([healthy_by_label.get(lbl, 0.0) / total_by_label[lbl] for lbl in label_ids])


def vectorize_upper_triangle(matrix_df, node_names):
    """Solo le connessioni uniche (triangolo superiore, diagonale esclusa), come vettore etichettato.

    La matrice FC e' simmetrica (A-B == B-A) e ha 1.0 sulla diagonale (un nodo correlato con se stesso) -
    tenerla intera duplicherebbe ogni valore e aggiungerebbe una diagonale non informativa.
    """
    n = len(node_names)
    row_idx, col_idx = np.triu_indices(n, k=1)  # k=1: esclude la diagonale
    edge_names = [f"{node_names[i]}__{node_names[j]}" for i, j in zip(row_idx, col_idx)]
    values = matrix_df.values[row_idx, col_idx]
    return pd.Series(values, index=edge_names)


def process_subject(subject):
    """Fase A completa per un paziente: lesione -> coverage -> masking (NaN) -> salvataggio -> vettorizzazione.

    Ritorna (edge_vector, n_compromised): il vettore vettorizzato e quante zone sono state marcate NaN,
    per il report di qualita' della Fase B.
    """
    lesion_path = (
        f"{DATA_ROOT}/manual_masks/{subject}/anat/"
        f"{subject}_space-MNI152NLin6Asym_label-lesion_mask.nii.gz"
    )
    lesion_img = nib.load(lesion_path)
    # stessa shape non implica stesso orientamento spaziale (vedi affine invertito, sezione successiva) -
    # resample esplicito via affine, mai un allineamento assunto.
    lesion_resampled = resample_to_img(
        lesion_img, atlas_img, interpolation="nearest", force_resample=True, copy_header=True
    )
    lesion_data = (np.asarray(lesion_resampled.get_fdata()) > 0.5).astype(np.int32)

    healthy_img = nib.Nifti1Image((1 - lesion_data), atlas_img.affine, atlas_img.header)
    parcel_coverage = compute_parcel_coverage(atlas_img, label_ids, healthy_img)
    compromised = parcel_coverage < MIN_COVERAGE
    compromised_names = node_names[compromised]

    fc_path = glob.glob(f"{DATA_ROOT}/features/{subject}/func/*{ATLAS_COMBO}*.csv")[0]
    fc = pd.read_csv(fc_path, sep="\t", index_col=0)
    assert list(fc.index) == list(node_names), f"{subject}: ordine nodi FC non combacia con l'atlante"

    fc_masked = fc.copy()  # non modifichiamo mai i dati originali
    fc_masked.loc[compromised_names, :] = np.nan  # NaN, non zero: il valore concreto si decide solo a valle
    fc_masked.loc[:, compromised_names] = np.nan

    # Salvataggio su disco della matrice mascherata COSI' COM'E' - prima di vettorizzare, formato ispezionabile.
    masked_path = MASKED_FC_ROOT / f"{subject}_masked_fc.csv"
    fc_masked.to_csv(masked_path)

    edge_vector = vectorize_upper_triangle(fc_masked, node_names)
    return edge_vector, len(compromised_names), masked_path


results = {}
for subject in SUBJECTS:
    edge_vector, n_compromised, masked_path = process_subject(subject)
    results[subject] = edge_vector
    print(f"{subject}: {n_compromised} zone compromesse su {len(node_names)} -> salvato in {masked_path}")

sub-STUNIPD0002: 0 zone compromesse su 239 -> salvato in ../data/derived/features/masked_fc/atlas-Yan200TianS2Buckner7N/sub-STUNIPD0002_masked_fc.csv
sub-STUNIPD0003: 8 zone compromesse su 239 -> salvato in ../data/derived/features/masked_fc/atlas-Yan200TianS2Buckner7N/sub-STUNIPD0003_masked_fc.csv
sub-STUNIPD0006: 9 zone compromesse su 239 -> salvato in ../data/derived/features/masked_fc/atlas-Yan200TianS2Buckner7N/sub-STUNIPD0006_masked_fc.csv


## Step 9-12 — tutti i pazienti insieme (`build_fc_matrix.py`)

Partono solo dopo che ogni paziente ha finito gli step 1-8 e ha prodotto il suo vettore in memoria. Non ha senso impilare o fare controllo qualità su un paziente solo.

9. Verifica che le etichette di connessione combacino tra tutti i pazienti (stesso ordine, stesse connessioni) — altrimenti errore, mai impilare "per posizione".
10. Impila: un paziente per riga → tabella pazienti × connessioni.
11. Controllo qualità: NaN per paziente (per riga, quanto è compromesso un paziente) e NaN per connessione (per colonna, quanto è affidabile una feature) — due domande diverse, vedi [Parte 2](../assets/knowledge/fc_lesion_masking.md#controllo-qualità-dopo-limpilamento) del `.md`.
12. Salva la matrice impilata su disco, ancora con i NaN dentro (output finale).

In [4]:
# 1. Verifica allineamento: tutti i pazienti devono avere le stesse identiche etichette di connessione.
reference_edges = results[SUBJECTS[0]].index
for subject, edge_vector in results.items():
    assert list(edge_vector.index) == list(reference_edges), f"{subject}: etichette di connessione disallineate"

# 2. Impila: un paziente per riga.
fc_matrix = pd.DataFrame({subject: vec for subject, vec in results.items()}).T
fc_matrix.index.name = "subject_id"
print(f"Matrice impilata: {fc_matrix.shape[0]} pazienti x {fc_matrix.shape[1]} connessioni\n")

# 3. Report di qualita' - mai calcolato e poi scartato.
nan_per_subject = fc_matrix.isna().sum(axis=1)
nan_per_edge = fc_matrix.isna().sum(axis=0)
print("NaN per paziente:")
print(nan_per_subject)
print(f"\nConnessioni mai compromesse in nessun paziente di questo campione: {(nan_per_edge == 0).sum()} / {len(nan_per_edge)}")
print(f"Connessione con piu' NaN nel campione: '{nan_per_edge.idxmax()}' ({nan_per_edge.max()} pazienti su {len(SUBJECTS)})")

# 4. Salvataggio della matrice impilata, ancora con i NaN - destinazione concordata.
fc_matrix_path = FC_MATRIX_ROOT / "fc_matrix_demo.csv"
fc_matrix.to_csv(fc_matrix_path)
print(f"\nMatrice impilata salvata in: {fc_matrix_path}")

Matrice impilata: 3 pazienti x 28441 connessioni

NaN per paziente:
subject_id
sub-STUNIPD0002       0
sub-STUNIPD0003    1876
sub-STUNIPD0006    2335
dtype: int64

Connessioni mai compromesse in nessun paziente di questo campione: 24310 / 28441
Connessione con piu' NaN nel campione: '7Networks_LH_SalVentAttn_Ins_1__7Networks_RH_Cont_PFCl_4' (2 pazienti su 3)

Matrice impilata salvata in: ../data/derived/features/fc_matrix/atlas-Yan200TianS2Buckner7N/fc_matrix_demo.csv


## Imputazione (anteprima, non parte della procedura)

La procedura finisce allo step 12: la matrice impilata resta con i NaN. L'imputazione vera (NaN → 0 o altra strategia) è un passo separato e successivo, da fare solo subito prima di un metodo che non tollera NaN (PCA/UMAP), vicino a `dim_reduction.py` — vedi decisione 2 in [Parte 2](../assets/knowledge/fc_lesion_masking.md#decisioni-di-metodo-prese) del `.md`.